In [191]:
import pandas as pd

In [192]:
!wget https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb
%run db2.ipynb
db2creds_file = 'db2con.env'
from dotenv import dotenv_values
db2creds = dotenv_values(db2creds_file)

--2024-06-05 03:46:07--  https://raw.githubusercontent.com/IBM/db2-jupyter/master/db2.ipynb


Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 156798 (153K) [text/plain]
Saving to: ‘db2.ipynb.16’

db2.ipynb.16        100%[===================>] 153.12K  --.-KB/s    in 0.01s   

2024-06-05 03:46:07 (12.5 MB/s) - ‘db2.ipynb.16’ saved [156798/156798]

Db2 Extensions Loaded. Version: 2024-05-29


In [193]:
%sql CONNECT CREDENTIALS db2creds

Connection successful. tpcds @ localhost 


In [194]:
df_queries_columns = ['query_id', 'appl_id', 'uow_id', 'activity_id', 'explain_time', 'query']

In [195]:
df_queries = pd.read_csv('success.csv', header=None, names=df_queries_columns)

In [196]:
df_queries.shape

(5764, 6)

In [197]:
df_queries.head(5)

,query_id,appl_id,uow_id,activity_id,explain_time,query
0,1,*LOCAL.shaikhq.240530151621,4,1,2024-05-30-08.16.11.238384,"SELECT TPCDS.CUSTOMER.C_BIRTH_YEAR , TPCDS.DAT..."
1,2,*LOCAL.shaikhq.240530151621,12,1,2024-05-30-08.16.12.871381,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
2,4,*LOCAL.shaikhq.240530151621,24,1,2024-05-30-08.16.24.574183,"SELECT TPCDS.WEB_SITE.WEB_CLOSE_DATE_SK , TPCD..."
3,5,*LOCAL.shaikhq.240530151621,32,1,2024-05-30-08.16.25.771665,"SELECT TPCDS.CATALOG_PAGE.CP_END_DATE_SK , TPC..."
4,6,*LOCAL.shaikhq.240530151621,40,1,2024-05-30-08.16.26.997895,"SELECT TPCDS.CATALOG_PAGE.CP_CATALOG_PAGE_SK ,..."


In [198]:
query1_ts = "2024-05-30-08.16.11.238384"
df_queries = df_queries[df_queries['explain_time'] == query1_ts]
query_id = df_queries['query_id'].values[0]
activity_id = df_queries['activity_id'].values[0]
appl_id = df_queries['appl_id'].values[0]
uow_id = df_queries['uow_id'].values[0]

In [199]:
query_id

1

# collecting query level stats

In [200]:
# collecting final actual card
sql = f""" 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

In [201]:
print(sql)

 
SELECT ROWS_RETURNED, 
    SORT_SHRHEAP_TOP 
FROM ACTIVITY_DB2ACTIVITIES 
WHERE ACTIVITY_ID = 1 AND 
APPL_ID = '*LOCAL.shaikhq.240530151621' AND 
UOW_ID = 4



In [202]:

df_activity = %sql {sql}

In [203]:

actual_card = df_activity['ROWS_RETURNED'].values[0]
sort_shrheap_top = df_activity['SORT_SHRHEAP_TOP'].values[0]

In [204]:
sql = f""" 
SELECT STMT_EXEC_TIME 
FROM ACTIVITYMETRICS_DB2ACTIVITIES 
WHERE ACTIVITY_ID = {activity_id} AND 
APPL_ID = '{appl_id}' AND 
UOW_ID = {uow_id}
"""

df_activitymetrics = %sql {sql}
stmt_exec_time = df_activitymetrics['STMT_EXEC_TIME'].values[0]

In [205]:
print('actual_card: {}'.format(actual_card))
print('sort_shrheap_top: {}'.format(sort_shrheap_top))
print('stmt_exec_time: {}'.format(stmt_exec_time))

actual_card: 8159
sort_shrheap_top: 69
stmt_exec_time: 365


# Collecting Node level information for each node

## Get the list of operator_ids for the current query

In [206]:
# find out the the nodes / operators
# fetching operators
sql = f"""
SELECT OPERATOR_ID, OPERATOR_TYPE 
FROM EXPLAIN_OPERATOR
WHERE EXPLAIN_TIME = '{query1_ts}'
"""

df_explain_operator = %sql {sql}
print(df_explain_operator)
operator_ids = df_explain_operator['OPERATOR_ID'].tolist()
print(operator_ids)

   OPERATOR_ID OPERATOR_TYPE
0            1        RETURN
1            2        HSJOIN
2            3        TBSCAN
3            4        TBSCAN
[1, 2, 3, 4]


In [207]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')
stream_cols = ['SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'COLUMN_NAMES']
df_explain_stream = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == query1_ts)][stream_cols]

In [208]:
# df_explain_stream.head()

In [209]:
# df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')

In [210]:
# df_explain_predicate['HOW_APPLIED'].unique()

In [211]:
df_explain_predicate = pd.read_csv('EXPLAIN_PREDICATE.csv')
predicate_cols = [ 'OPERATOR_ID',
       'PREDICATE_ID', 'HOW_APPLIED', 'WHEN_EVALUATED', 'RELOP_TYPE',
       'SUBQUERY', 'FILTER_FACTOR', 'PREDICATE_TEXT']

df_predicate_filtered = df_explain_predicate[df_explain_predicate['EXPLAIN_TIME'] == query1_ts][predicate_cols]
print('df_predicate_filtered shape :{}'.format(df_predicate_filtered.shape))
print('df_predicate_filtered: ', df_predicate_filtered)

df_predicate_filtered shape :(3, 8)
df_predicate_filtered:     OPERATOR_ID  PREDICATE_ID HOW_APPLIED WHEN_EVALUATED RELOP_TYPE SUBQUERY  \
0            2             2  JOIN                              EQ        N   
1            4             3  SARG                              LE        N   
2            4             4  SARG                              EQ        N   

   FILTER_FACTOR                              PREDICATE_TEXT  
0       0.000014  (Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)  
1       0.713558                         (1958 <= Q2.D_YEAR)  
2       0.081000                             (Q2.D_MOY = 12)  


In [212]:
# df_predicate_filtered

In [213]:
# op_id = 3
# df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == op_id]['PREDICATE_TEXT'].values

In [214]:
# df_explain_stream.shape

In [215]:
# print(df_explain_stream['COLUMN_NAMES'][0])

In [216]:
nodes = {}

for operator_id in operator_ids:
    node_dict = {}
    node_dict['Node Type'] = df_explain_operator[df_explain_operator['OPERATOR_ID'] == operator_id]['OPERATOR_TYPE'].values[0]
    
    # check if there is any relation / table involved in this operation
    relation_name = df_explain_stream[(df_explain_stream['TARGET_ID'] == operator_id) 
                                      & (df_explain_stream['OBJECT_NAME'].notna())]['OBJECT_NAME'].values
    
    if len(relation_name) > 0:
        node_dict['Relation Name'] = relation_name[0]
        # collecting local predicates, if any
        local_predicate = df_predicate_filtered[df_predicate_filtered['OPERATOR_ID'] == operator_id]['PREDICATE_TEXT'].values
        if len(local_predicate) > 0:
            node_dict['Filter'] = ' AND '.join(local_predicate.tolist())
        
    # check if there is any join predicate
    join_predicate = df_predicate_filtered[(df_predicate_filtered['OPERATOR_ID'] == operator_id) & 
                          (df_predicate_filtered['HOW_APPLIED'].str.strip() == 'JOIN')]['PREDICATE_TEXT'].values
    
    if len(join_predicate) > 0:
        node_dict['Join Predicate'] = join_predicate[0]
    
    nodes[operator_id] = node_dict


In [217]:
nodes

{1: {'Node Type': 'RETURN'},
 2: {'Node Type': 'HSJOIN',
  'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)'},
 3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}}

In [218]:
df_explain_stream

,SOURCE_ID,TARGET_TYPE,TARGET_ID,OBJECT_NAME,STREAM_COUNT,COLUMN_COUNT,COLUMN_NAMES
0,-1,O,3,CUSTOMER,100000.000000,3,+Q1.$RID$+Q1.C_BIRTH_YEAR+Q1.C_FIRST_SHIPTO_DA...
1,3,O,2,NaN,100000.000000,-1,NaN
2,-1,O,4,DATE_DIM2,73049.000000,5,+Q2.$RID$+Q2.D_WEEK_SEQ+Q2.D_YEAR+Q2.D_MOY+Q2....
3,4,O,2,NaN,3983.533936,-1,NaN
4,2,O,1,NaN,104386.367188,-1,NaN


In [219]:

# Identifying parent-child relationship
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[stream_cols]

In [220]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN'}
2: {'Node Type': 'HSJOIN', 'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)'}
3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'}
4: {'Node Type': 'TBSCAN', 'Relation Name': 'DATE_DIM2', 'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}


# Scratchpad section

In [221]:
df_explain_stream = pd.read_csv('EXPLAIN_STREAM.csv')

In [222]:
df_explain_stream.columns

Index(['EXPLAIN_REQUESTER', 'EXPLAIN_TIME', 'SOURCE_NAME', 'SOURCE_SCHEMA',
       'SOURCE_VERSION', 'EXPLAIN_LEVEL', 'STMTNO', 'SECTNO', 'STREAM_ID',
       'SOURCE_TYPE', 'SOURCE_ID', 'TARGET_TYPE', 'TARGET_ID', 'OBJECT_SCHEMA',
       'OBJECT_NAME', 'STREAM_COUNT', 'COLUMN_COUNT', 'PREDICATE_ID',
       'COLUMN_NAMES', 'PMID', 'SINGLE_NODE', 'PARTITION_COLUMNS',
       'SEQUENCE_SIZES', 'OBJECT_TENANTID'],
      dtype='object')

In [223]:
stream_cols = ['SOURCE_ID', 'TARGET_ID']
df_stream_filtered = df_explain_stream[(df_explain_stream['EXPLAIN_TIME'] == query1_ts)][stream_cols]

In [224]:
nodes

{1: {'Node Type': 'RETURN'},
 2: {'Node Type': 'HSJOIN',
  'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)'},
 3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}}

In [225]:
for index, row in df_stream_filtered.iterrows():
    source_id = row['SOURCE_ID']
    target_id = row['TARGET_ID']
    if source_id > 0:
        if 'Plans' not in nodes[target_id]:
            nodes[target_id]['Plans'] = []
        nodes[target_id]['Plans'].append(source_id)
    #print('source: {}, target: {}'.format(source_id, target_id))

In [226]:
for key, value in nodes.items():
    print(f'{key}: {value}')

1: {'Node Type': 'RETURN', 'Plans': [2]}
2: {'Node Type': 'HSJOIN', 'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)', 'Plans': [3, 4]}
3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'}
4: {'Node Type': 'TBSCAN', 'Relation Name': 'DATE_DIM2', 'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}


In [227]:
nodes

{1: {'Node Type': 'RETURN', 'Plans': [2]},
 2: {'Node Type': 'HSJOIN',
  'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)',
  'Plans': [3, 4]},
 3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}}

In [228]:
json_df = pd.DataFrame(columns=['id', 'json'])

In [229]:
nodes

{1: {'Node Type': 'RETURN', 'Plans': [2]},
 2: {'Node Type': 'HSJOIN',
  'Join Predicate': '(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)',
  'Plans': [3, 4]},
 3: {'Node Type': 'TBSCAN', 'Relation Name': 'CUSTOMER'},
 4: {'Node Type': 'TBSCAN',
  'Relation Name': 'DATE_DIM2',
  'Filter': '(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)'}}

In [230]:
import json

def build_tree(nodes, node_key):
    node = nodes[node_key].copy()  # Get the node and make a copy of it
    if 'Plans' in node:  # If the node has children
        node['Plans'] = [build_tree(nodes, child_key) for child_key in node['Plans']]  # Replace child keys with child nodes
    return node

tree = {"Plan": build_tree(nodes, 1)}
tree['stmt_exec_time'] = float(stmt_exec_time)
tree['sort_shrheap_top'] = float(sort_shrheap_top)
json_object = json.dumps(tree)

print(json_object)

{"Plan": {"Node Type": "RETURN", "Plans": [{"Node Type": "HSJOIN", "Join Predicate": "(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)", "Plans": [{"Node Type": "TBSCAN", "Relation Name": "CUSTOMER"}, {"Node Type": "TBSCAN", "Relation Name": "DATE_DIM2", "Filter": "(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)"}]}]}, "stmt_exec_time": 365.0, "sort_shrheap_top": 69.0}


In [231]:
json_df.loc[len(json_df)] = [query_id, json_object]

In [232]:
json_df

,id,json
0,1,"{""Plan"": {""Node Type"": ""RETURN"", ""Plans"": [{""N..."


In [233]:
json_parsed = json.loads(json_df['json'].iloc[0])
json_pretty = json.dumps(json_parsed, indent=4)
print(json_pretty)

with open('output.json', 'w') as f:
    f.write(json_pretty)

{
    "Plan": {
        "Node Type": "RETURN",
        "Plans": [
            {
                "Node Type": "HSJOIN",
                "Join Predicate": "(Q2.D_DATE_SK = Q1.C_FIRST_SHIPTO_DATE_SK)",
                "Plans": [
                    {
                        "Node Type": "TBSCAN",
                        "Relation Name": "CUSTOMER"
                    },
                    {
                        "Node Type": "TBSCAN",
                        "Relation Name": "DATE_DIM2",
                        "Filter": "(1958 <= Q2.D_YEAR) AND (Q2.D_MOY = 12)"
                    }
                ]
            }
        ]
    },
    "stmt_exec_time": 365.0,
    "sort_shrheap_top": 69.0
}


In [38]:
type(nodes)

dict

In [39]:
type(nodes[1])

dict